# 01.3 — Data Preparation for All Holdout Experiments (Renorm Pipeline)

Produces the same 4-groups-x-3-files dataset layout as [07_data_prep_all_holdouts.ipynb](07_data_prep_all_holdouts.ipynb), but using the re-normalized preprocessing pipeline:

- Raw integer counts loaded from `.raw` (both species)
- `normalize_total(target_sum=1e4)` + `log1p` applied identically to mouse and human
- HVG selection with `batch_key="species"` (instead of cross-species merged HVG)
- Same ortholog alignment + cell matching logic as 07

**Groups**:

| Group | Holdout (ontology IDs) | Also excluded from AE | Description |
|-------|------------------------|----------------------|-------------|
| A | CD8 (CL:0000625) | — | CD8 holdout only |
| B | CD8 (CL:0000625) | thymocytes (CL:0000893) | CD8 + thymocyte excluded |
| C | CD4+CD8+thymocytes | — | All T-cell subtypes |
| D | CD4 (CL:0000624) | — | Different T-cell subtype from A |

**Each group produces 3 files**:

| File | Used by model | Semantics |
|---|---|---|
| `ae_training_<group>_renorm.h5ad` | **scGen** | All cells minus excluded; AE training data |
| `<group>_holdout_renorm.h5ad` | **swapped CellOT** (= IMPACT in repo) | Matched pairs, `condition=species`; mouse -> human transport, cell type held out as OOD |
| `<group>_holdout_swapped_renorm.h5ad` | **Normal CellOT** (CellOT paper) | Matched pairs, `condition=cell_type_status`; non_X -> X transport, human species held out |

**Naming warning**: The `_swapped` suffix in the third filename refers to a CODE-level operation (the `condition` column was swapped from species to cell-type-status), NOT to the model we call "swapped CellOT." The `_swapped` file is used by **Normal CellOT**, not by swapped CellOT. See `research_log_2026-04-20.txt` for the full story.

**Output directory**: `/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-renorm/`

In [1]:
import sys
sys.path.insert(0, "/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT")

import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy import sparse as sp_sparse

from speciesot_helpers import (
    align_adatas_biomart_one2one,
    match_cells_by_celltype_tissue,
)

MOUSE_H5AD = "/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/data/tabula_muris/sampled_mouse_shared.h5ad"
HUMAN_H5AD = "/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/data/tabula_sapiens/sampled_human_shared.h5ad"

BASE_DIR = "/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT"
DATASET_DIR = os.path.join(BASE_DIR, "cellot/cellot_gpu/datasets/speciesot-human-mouse-renorm")
os.makedirs(DATASET_DIR, exist_ok=True)

CT_COL = "cell_type_ontology_term_id"
N_HVG = 1000

sc.settings.verbosity = 1
print("Imports OK")
print(f"Output directory: {DATASET_DIR}")

/n/home01/jzhou1125/miniforge3/envs/analysis/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK
Output directory: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-renorm


## 1. Load raw counts and re-normalize identically

Load the `sampled_*_shared.h5ad` files, promote `.raw` (integer UMI/read counts) to `.X`, then apply the same `normalize_total(target_sum=1e4)` + `log1p` to both species. This replaces the mismatched upstream `.X` (human log-normalized only; mouse log-normalized + per-gene /std + clipped) that notebook 07 consumes.

In [2]:
print("Loading source files and promoting .raw to .X ...")
mouse_full = sc.read_h5ad(MOUSE_H5AD)
human_full = sc.read_h5ad(HUMAN_H5AD)

assert mouse_full.raw is not None, "mouse_full.raw is missing; cannot proceed"
assert human_full.raw is not None, "human_full.raw is missing; cannot proceed"

print(f"  mouse_full: {mouse_full.shape}   .raw: {mouse_full.raw.shape}")
print(f"  human_full: {human_full.shape}   .raw: {human_full.raw.shape}")

mouse_all = mouse_full.raw.to_adata()
human_all = human_full.raw.to_adata()

# Preserve full obs metadata (raw.to_adata may narrow it in some configurations)
mouse_all.obs = mouse_full.obs.copy()
human_all.obs = human_full.obs.copy()

print("\nApplying normalize_total(target_sum=1e4) + log1p to both species identically ...")
for species_name, a in [("mouse_all", mouse_all), ("human_all", human_all)]:
    a.X = a.X.astype("float32")
    sc.pp.normalize_total(a, target_sum=1e4)
    sc.pp.log1p(a)
    d = a.X.data if sp_sparse.issparse(a.X) else a.X.ravel()
    print(f"  {species_name}: {a.shape}   min={d.min():.4f}  max={d.max():.4f}  mean={d.mean():.4f}")

print("\nBoth species now on identical log1p(normalize_total(1e4)) scale.")

Loading source files and promoting .raw to .X ...


  mouse_full: (47807, 18024)   .raw: (47807, 18024)
  human_full: (58931, 61759)   .raw: (58931, 61759)

Applying normalize_total(target_sum=1e4) + log1p to both species identically ...
  mouse_all: (47807, 18024)   min=0.0003  max=8.9909  mean=1.1866
  human_all: (58931, 61759)   min=0.0003  max=9.1015  mean=0.9594

Both species now on identical log1p(normalize_total(1e4)) scale.


## 2. Align orthologs (BioMart one-to-one)

Same function as 07 (`align_adatas_biomart_one2one`). Input: re-normalized full-gene mouse and human. Output: both AnnDatas with a shared ~14k gene axis of one-to-one orthologs.

In [3]:
print("Aligning orthologs via BioMart one-to-one ...")
mouse_all_aligned, human_all_aligned, ortholog_table = align_adatas_biomart_one2one(
    mouse_all, human_all
)

print(f"  Ortholog pairs: {len(ortholog_table)}")
print(f"  Mouse aligned: {mouse_all_aligned.shape}")
print(f"  Human aligned: {human_all_aligned.shape}")

# Tag condition=species for HVG / concat
mouse_all_aligned.obs["condition"] = "mouse"
human_all_aligned.obs["condition"] = "human"

Aligning orthologs via BioMart one-to-one ...
  Ortholog pairs: 14451
  Mouse aligned: (47807, 14451)
  Human aligned: (58931, 14451)


In [4]:
ortholog_table

,human_ensembl_id,human_gene_name,mouse_ensembl_id,mouse_gene_name,orthology_type
0,ENSG00000154620,TMSB4Y,ENSMUSG00000049775,Tmsb4x,ortholog_one2one
1,ENSG00000067048,DDX3Y,ENSMUSG00000069045,Ddx3y,ortholog_one2one
2,ENSG00000114374,USP9Y,ENSMUSG00000069044,Usp9y,ortholog_one2one
3,ENSG00000183878,UTY,ENSMUSG00000068457,Uty,ortholog_one2one
4,ENSG00000012817,KDM5D,ENSMUSG00000056673,Kdm5d,ortholog_one2one
...,...,...,...,...,...
14446,ENSG00000143740,SNAP47,ENSMUSG00000009894,Snap47,ortholog_one2one
14447,ENSG00000117472,TSPAN1,ENSMUSG00000028699,Tspan1,ortholog_one2one
14448,ENSG00000168710,AHCYL1,ENSMUSG00000027893,Ahcyl1,ortholog_one2one
14449,ENSG00000081692,JMJD4,ENSMUSG00000036819,Jmjd4,ortholog_one2one


## 3. HVG selection with `batch_key="species"`

Same HVG call as 07 but with `batch_key="species"` so per-species dispersion is computed independently and then merged. This prevents species-baseline variance from being mistaken for biological variance.

In [4]:
all_cells = ad.concat([mouse_all_aligned, human_all_aligned], join="inner")
all_cells.obs["species"] = all_cells.obs["condition"].values

print(f"  Concatenated all cells: {all_cells.shape}")

sc.pp.highly_variable_genes(
    all_cells,
    n_top_genes=N_HVG,
    flavor="seurat",
    batch_key="species",
)

hvg_genes = all_cells.var_names[all_cells.var.highly_variable].tolist()
n_intersection = int(all_cells.var["highly_variable_intersection"].sum())
n_nbatches_2 = int((all_cells.var["highly_variable_nbatches"] == 2).sum())

print(f"  Selected {len(hvg_genes)} HVGs (batch_key='species')")
print(f"  Of which flagged in BOTH species (nbatches=2): {n_nbatches_2}")
print(f"  Intersection column sum (same as above): {n_intersection}")

# Subset both species to the HVG columns
mouse_all_hvg = mouse_all_aligned[:, hvg_genes].copy()
human_all_hvg = human_all_aligned[:, hvg_genes].copy()
mouse_all_hvg.obs["condition"] = "mouse"
human_all_hvg.obs["condition"] = "human"

print(f"\n  Mouse HVG: {mouse_all_hvg.shape}")
print(f"  Human HVG: {human_all_hvg.shape}")

  Concatenated all cells: (106738, 14451)
  Selected 1000 HVGs (batch_key='species')
  Of which flagged in BOTH species (nbatches=2): 269
  Intersection column sum (same as above): 269

  Mouse HVG: (47807, 1000)
  Human HVG: (58931, 1000)


## 4. Match cells by `(cell_type, tissue)`

Identical to 07: `match_cells_by_celltype_tissue` on the HVG-subset data. Returns `min(n_mouse, n_human)` cells per `(cell_type, tissue)` identity for each species.

In [5]:
mouse_matched_hvg, human_matched_hvg = match_cells_by_celltype_tissue(
    mouse_all_hvg, human_all_hvg,
    cell_type_key=CT_COL,
    tissue_key="tissue_ontology_term_id",
)

print(f"  Matched mouse: {mouse_matched_hvg.shape}")
print(f"  Matched human: {human_matched_hvg.shape}")

  Matched mouse: (6495, 1000)
  Matched human: (6495, 1000)


## 5. Define holdout groups and utility functions

Identical to 07 **plus Group A** (which 07 delegated to notebook 05) and **Group B renamed** from `cd8_nothymo` to `cd8_thymo`.

In [6]:
keep_obs = [
    "condition", "species",
    "cell_type_ontology_term_id", "cell_type",
    "tissue_ontology_term_id", "tissue",
    "donor_id",
]


def clean_adata(adata):
    """Strip layers/obsm/uns and densify X for compatibility with older anndata in CellOT env."""
    obs_cols = [c for c in keep_obs if c in adata.obs.columns]
    X = adata.X
    if sp_sparse.issparse(X):
        X = np.array(X.todense())
    elif not isinstance(X, np.ndarray):
        X = np.array(X)
    return ad.AnnData(
        X=X.astype(np.float32),
        obs=adata.obs[obs_cols].copy(),
        var=pd.DataFrame(index=adata.var_names),
    )


GROUPS = {
    "A": {
        "name": "cd8",
        "description": "CD8 holdout only",
        "holdout_ids": ["CL:0000625"],
        "exclude_from_ae": ["CL:0000625"],
        "holdout_label": "cd8",
    },
    "B": {
        "name": "cd8_thymo",
        "description": "CD8 holdout; thymocytes also excluded from AE",
        "holdout_ids": ["CL:0000625"],
        "exclude_from_ae": ["CL:0000625", "CL:0000893"],
        "holdout_label": "cd8",
    },
    "C": {
        "name": "tcell_subtypes",
        "description": "All T-cell subtypes held out (CD4+CD8+thymocytes)",
        "holdout_ids": ["CL:0000624", "CL:0000625", "CL:0000893"],
        "exclude_from_ae": ["CL:0000624", "CL:0000625", "CL:0000893"],
        "holdout_label": "tcell_subtype",
    },
    "D": {
        "name": "cd4",
        "description": "CD4 holdout only",
        "holdout_ids": ["CL:0000624"],
        "exclude_from_ae": ["CL:0000624"],
        "holdout_label": "cd4",
    },
}

print("Holdout groups defined:")
for gid, g in GROUPS.items():
    print(f"  Group {gid} ({g['name']}): {g['description']}")
    print(f"    holdout_ids:     {g['holdout_ids']}")
    print(f"    exclude_from_ae: {g['exclude_from_ae']}")

Holdout groups defined:
  Group A (cd8): CD8 holdout only
    holdout_ids:     ['CL:0000625']
    exclude_from_ae: ['CL:0000625']
  Group B (cd8_thymo): CD8 holdout; thymocytes also excluded from AE
    holdout_ids:     ['CL:0000625']
    exclude_from_ae: ['CL:0000625', 'CL:0000893']
  Group C (tcell_subtypes): All T-cell subtypes held out (CD4+CD8+thymocytes)
    holdout_ids:     ['CL:0000624', 'CL:0000625', 'CL:0000893']
    exclude_from_ae: ['CL:0000624', 'CL:0000625', 'CL:0000893']
  Group D (cd4): CD4 holdout only
    holdout_ids:     ['CL:0000624']
    exclude_from_ae: ['CL:0000624']


## 6. Inspect cell counts per group

In [7]:
T_CELL_IDS = {
    "CL:0000084": "T cell (broad)",
    "CL:0000893": "thymocyte",
    "CL:0000624": "CD4+ T cell",
    "CL:0000625": "CD8+ T cell",
}

matched_hvg = ad.concat([mouse_matched_hvg, human_matched_hvg], join="inner")
all_hvg = ad.concat([mouse_all_hvg, human_all_hvg], join="inner")

print("T cell family in matched dataset:")
print("=" * 70)
for cid, name in T_CELL_IDS.items():
    mask = matched_hvg.obs[CT_COL].astype(str) == cid
    n_total = int(mask.sum())
    n_mouse = int((mask & (matched_hvg.obs["condition"] == "mouse")).sum())
    n_human = int((mask & (matched_hvg.obs["condition"] == "human")).sum())
    print(f"  {cid} ({name}): {n_total} total ({n_mouse} mouse, {n_human} human)")

print(f"\nTotal matched cells: {matched_hvg.n_obs}")
print(f"Total all cells: {all_hvg.n_obs}")

for gid, g in GROUPS.items():
    excluded = set(g["exclude_from_ae"])
    ae_mask = ~all_hvg.obs[CT_COL].astype(str).isin(excluded)
    holdout_mask = matched_hvg.obs[CT_COL].astype(str).isin(g["holdout_ids"])
    print(f"\nGroup {gid} ({g['name']}):")
    print(f"  AE training cells: {int(ae_mask.sum())} (excluded {int((~ae_mask).sum())})")
    print(f"  Holdout cells in matched data: {int(holdout_mask.sum())}")

T cell family in matched dataset:
  CL:0000084 (T cell (broad)): 204 total (102 mouse, 102 human)
  CL:0000893 (thymocyte): 910 total (455 mouse, 455 human)
  CL:0000624 (CD4+ T cell): 192 total (96 mouse, 96 human)
  CL:0000625 (CD8+ T cell): 390 total (195 mouse, 195 human)

Total matched cells: 12990
Total all cells: 106738

Group A (cd8):
  AE training cells: 104738 (excluded 2000)
  Holdout cells in matched data: 390

Group B (cd8_thymo):
  AE training cells: 103283 (excluded 3455)
  Holdout cells in matched data: 390

Group C (tcell_subtypes):
  AE training cells: 101283 (excluded 5455)
  Holdout cells in matched data: 1492

Group D (cd4):
  AE training cells: 104738 (excluded 2000)
  Holdout cells in matched data: 192


## 7. Generate datasets for each group

For each group, write 3 files (see top-of-notebook table). File naming uses the `_renorm` infix to distinguish from the stale-preprocessing outputs from notebook 07.

In [8]:
for gid, g in GROUPS.items():
    group_name = g["name"]
    holdout_ids = set(g["holdout_ids"])
    exclude_ids = set(g["exclude_from_ae"])
    label = g["holdout_label"]

    print(f"\n{'=' * 70}")
    print(f"GROUP {gid}: {g['description']}")
    print(f"{'=' * 70}")

    # --- File 1: AE training (scGen) ---
    ae_all = ad.concat([mouse_all_hvg, human_all_hvg], join="inner")
    ae_mask = ~ae_all.obs[CT_COL].astype(str).isin(exclude_ids)
    ae_data = clean_adata(ae_all[ae_mask].copy())

    ae_path = os.path.join(DATASET_DIR, f"ae_training_{group_name}_renorm.h5ad")
    ae_data.write_h5ad(ae_path)
    print(f"  AE training (scGen):  {ae_data.n_obs} cells -> {os.path.basename(ae_path)}")
    print(f"    condition counts: {dict(ae_data.obs['condition'].value_counts())}")

    # --- File 2: swapped CellOT data (condition=species, mouse -> human) ---
    mouse_m = mouse_matched_hvg.copy()
    human_m = human_matched_hvg.copy()
    mouse_m.obs["condition"] = "mouse"
    human_m.obs["condition"] = "human"
    swapcellot_data = ad.concat([mouse_m, human_m], join="inner")
    swapcellot_data = clean_adata(swapcellot_data)

    swapcellot_path = os.path.join(DATASET_DIR, f"{group_name}_holdout_renorm.h5ad")
    swapcellot_data.write_h5ad(swapcellot_path)
    n_holdout = int(swapcellot_data.obs[CT_COL].astype(str).isin(holdout_ids).sum())
    print(f"  swapped CellOT (condition=species): {swapcellot_data.n_obs} cells, "
          f"{n_holdout} OOD holdout -> {os.path.basename(swapcellot_path)}")

    # --- File 3: Normal CellOT data (condition=cell_type_status, holdout=human species) ---
    mouse_s = mouse_matched_hvg.copy()
    human_s = human_matched_hvg.copy()
    normalcellot = ad.concat([mouse_s, human_s], join="inner")
    normalcellot.obs["species"] = normalcellot.obs["condition"].values
    is_holdout = normalcellot.obs[CT_COL].astype(str).isin(holdout_ids)
    normalcellot.obs["condition"] = np.where(is_holdout, label, f"non_{label}")
    normalcellot = clean_adata(normalcellot)

    normalcellot_path = os.path.join(DATASET_DIR, f"{group_name}_holdout_swapped_renorm.h5ad")
    normalcellot.write_h5ad(normalcellot_path)
    print(f"  Normal CellOT  (condition={label}/non_{label}): "
          f"{normalcellot.n_obs} cells -> {os.path.basename(normalcellot_path)}")
    print(f"    condition values: {dict(normalcellot.obs['condition'].value_counts())}")
    print(f"    species values:   {dict(normalcellot.obs['species'].value_counts())}")

print(f"\n{'=' * 70}")
print("All datasets generated.")


GROUP A: CD8 holdout only
  AE training (scGen):  104738 cells -> ae_training_cd8_renorm.h5ad
    condition counts: {'human': np.int64(57931), 'mouse': np.int64(46807)}
  swapped CellOT (condition=species): 12990 cells, 390 OOD holdout -> cd8_holdout_renorm.h5ad
  Normal CellOT  (condition=cd8/non_cd8): 12990 cells -> cd8_holdout_swapped_renorm.h5ad
    condition values: {'non_cd8': np.int64(12600), 'cd8': np.int64(390)}
    species values:   {'human': np.int64(6495), 'mouse': np.int64(6495)}

GROUP B: CD8 holdout; thymocytes also excluded from AE
  AE training (scGen):  103283 cells -> ae_training_cd8_thymo_renorm.h5ad
    condition counts: {'human': np.int64(57476), 'mouse': np.int64(45807)}
  swapped CellOT (condition=species): 12990 cells, 390 OOD holdout -> cd8_thymo_holdout_renorm.h5ad
  Normal CellOT  (condition=cd8/non_cd8): 12990 cells -> cd8_thymo_holdout_swapped_renorm.h5ad
    condition values: {'non_cd8': np.int64(12600), 'cd8': np.int64(390)}
    species values:   {'huma

## 8. Create `_v07` copies (for CellOT env compatibility)

The CellOT conda environment uses an older anndata version that expects the v0.7 format. `clean_adata()` already strips incompatible fields, so the same file should work. We still write explicit `_v07` copies to match the existing task-YAML naming convention.

In [9]:
import shutil

for gid, g in GROUPS.items():
    group_name = g["name"]
    for base in [
        f"ae_training_{group_name}_renorm",
        f"{group_name}_holdout_renorm",
        f"{group_name}_holdout_swapped_renorm",
    ]:
        src = os.path.join(DATASET_DIR, f"{base}.h5ad")
        dst = os.path.join(DATASET_DIR, f"{base}_v07.h5ad")
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copy2(src, dst)
            print(f"  Copied {os.path.basename(src)} -> {os.path.basename(dst)}")

print("\n_v07 copies created.")

  Copied ae_training_cd8_renorm.h5ad -> ae_training_cd8_renorm_v07.h5ad
  Copied cd8_holdout_renorm.h5ad -> cd8_holdout_renorm_v07.h5ad
  Copied cd8_holdout_swapped_renorm.h5ad -> cd8_holdout_swapped_renorm_v07.h5ad
  Copied ae_training_cd8_thymo_renorm.h5ad -> ae_training_cd8_thymo_renorm_v07.h5ad
  Copied cd8_thymo_holdout_renorm.h5ad -> cd8_thymo_holdout_renorm_v07.h5ad
  Copied cd8_thymo_holdout_swapped_renorm.h5ad -> cd8_thymo_holdout_swapped_renorm_v07.h5ad
  Copied ae_training_tcell_subtypes_renorm.h5ad -> ae_training_tcell_subtypes_renorm_v07.h5ad
  Copied tcell_subtypes_holdout_renorm.h5ad -> tcell_subtypes_holdout_renorm_v07.h5ad
  Copied tcell_subtypes_holdout_swapped_renorm.h5ad -> tcell_subtypes_holdout_swapped_renorm_v07.h5ad
  Copied ae_training_cd4_renorm.h5ad -> ae_training_cd4_renorm_v07.h5ad
  Copied cd4_holdout_renorm.h5ad -> cd4_holdout_renorm_v07.h5ad
  Copied cd4_holdout_swapped_renorm.h5ad -> cd4_holdout_swapped_renorm_v07.h5ad

_v07 copies created.


## 9. Rewrite `_v07` files for CellOT-env `anndata 0.7` compatibility

The `_v07` copies made above were saved by the current (newer) `anndata`, which is incompatible with the CellOT training environment's `anndata 0.7` in two ways:

1. **Empty HDF5 groups.** Newer `anndata` writes empty placeholder groups (`/layers`, `/obsm`, `/obsp`, `/uns`, `/varm`, `/varp`) and top-level `encoding-type` / `encoding-version` attributes. `anndata 0.7` does not expect these and raises `AnnDataReadError`.
2. **Categorical columns.** Newer `anndata` serializes `.obs` categoricals as nested groups with `categories` + `codes` (encoding-version 0.2.0). `anndata 0.7` expects a flat Dataset with integer codes and a shared `__categories` group (encoding-version 0.1.0).

Fix: open each `_v07.h5ad` via version-agnostic `h5py` to strip the empty groups, then round-trip it through the **CellOT environment's own Python interpreter** so `anndata 0.7` re-serializes `.obs` in the 0.1.0 format. After this step, the `_v07` files are readable by `cellot/cellot_gpu/scripts/train.py` and `evaluate.py`.

In [ ]:
import subprocess

CELLOT_PY = "/n/home01/jzhou1125/.conda/envs/CellOT/bin/python"

v07_files = sorted(
    os.path.join(DATASET_DIR, f)
    for f in os.listdir(DATASET_DIR)
    if f.endswith("_v07.h5ad")
)
print(f"Post-processing {len(v07_files)} _v07 files for anndata 0.7 compatibility ...")

strip_script = r"""
import sys, h5py

EMPTY_GROUPS = ["layers", "obsm", "obsp", "uns", "varm", "varp"]
for path in sys.argv[1:]:
    with h5py.File(path, "r+") as f:
        for g in EMPTY_GROUPS:
            if g in f and len(f[g].keys()) == 0:
                del f[g]
        for attr in ("encoding-type", "encoding-version"):
            if attr in f.attrs:
                del f.attrs[attr]
    print("  stripped:", path)
"""

subprocess.run(
    [CELLOT_PY, "-c", strip_script, *v07_files],
    check=True,
)

rewrite_script = r"""
import sys, os
import h5py
import numpy as np
import pandas as pd
import anndata as ad
from scipy import sparse

def load_obs(f):
    obs_grp = f["obs"]
    cols = {}
    if "_index" in obs_grp.attrs:
        idx_key = obs_grp.attrs["_index"].decode() if isinstance(obs_grp.attrs["_index"], bytes) else obs_grp.attrs["_index"]
    else:
        idx_key = "index"
    index = [x.decode() if isinstance(x, bytes) else x for x in obs_grp[idx_key][:]]
    for name in obs_grp.keys():
        if name == idx_key:
            continue
        node = obs_grp[name]
        if isinstance(node, h5py.Group) and "categories" in node and "codes" in node:
            cats = [c.decode() if isinstance(c, bytes) else c for c in node["categories"][:]]
            codes = node["codes"][:]
            cols[name] = pd.Categorical.from_codes(codes, categories=cats)
        else:
            arr = node[:]
            if arr.dtype.kind == "O" or arr.dtype.kind == "S":
                arr = np.array([x.decode() if isinstance(x, bytes) else x for x in arr])
            cols[name] = arr
    return pd.DataFrame(cols, index=pd.Index(index, name=idx_key))

def load_var(f):
    var_grp = f["var"]
    if "_index" in var_grp.attrs:
        idx_key = var_grp.attrs["_index"].decode() if isinstance(var_grp.attrs["_index"], bytes) else var_grp.attrs["_index"]
    else:
        idx_key = "index"
    index = [x.decode() if isinstance(x, bytes) else x for x in var_grp[idx_key][:]]
    cols = {}
    for name in var_grp.keys():
        if name == idx_key:
            continue
        node = var_grp[name]
        if isinstance(node, h5py.Group) and "categories" in node and "codes" in node:
            cats = [c.decode() if isinstance(c, bytes) else c for c in node["categories"][:]]
            codes = node["codes"][:]
            cols[name] = pd.Categorical.from_codes(codes, categories=cats)
        else:
            arr = node[:]
            if arr.dtype.kind == "O" or arr.dtype.kind == "S":
                arr = np.array([x.decode() if isinstance(x, bytes) else x for x in arr])
            cols[name] = arr
    return pd.DataFrame(cols, index=pd.Index(index, name=idx_key))

def load_X(f):
    X_node = f["X"]
    if isinstance(X_node, h5py.Group):
        data = X_node["data"][:]
        indices = X_node["indices"][:]
        indptr = X_node["indptr"][:]
        shape = tuple(X_node.attrs["shape"]) if "shape" in X_node.attrs else tuple(X_node.attrs["h5sparse_shape"])
        encoding = X_node.attrs.get("encoding-type", b"csr_matrix")
        if isinstance(encoding, bytes):
            encoding = encoding.decode()
        if "csc" in encoding:
            return sparse.csc_matrix((data, indices, indptr), shape=shape)
        return sparse.csr_matrix((data, indices, indptr), shape=shape)
    return X_node[:]

for path in sys.argv[1:]:
    with h5py.File(path, "r") as f:
        obs_df = load_obs(f)
        var_df = load_var(f)
        X = load_X(f)
    adata = ad.AnnData(X=X, obs=obs_df, var=var_df)
    os.remove(path)
    adata.write(path)
    print("  rewrote:", path, "| anndata", ad.__version__, "| shape", adata.shape)
"""

subprocess.run(
    [CELLOT_PY, "-c", rewrite_script, *v07_files],
    check=True,
)

print("\nAll _v07 files are now CellOT-env compatible.")

## 10. Verification

Lists files, checks all expected outputs exist, and performs a quick sanity check on one AE training file and one holdout file to confirm the re-normalized scale.

In [10]:
print("Files in output directory:")
print("=" * 70)
files_in_dir = sorted(f for f in os.listdir(DATASET_DIR) if f.endswith(".h5ad"))
for f in files_in_dir:
    fpath = os.path.join(DATASET_DIR, f)
    size_mb = os.path.getsize(fpath) / 1024 / 1024
    print(f"  {f:60s} {size_mb:6.1f} MB")

expected = []
for gid, g in GROUPS.items():
    group_name = g["name"]
    for base in [
        f"ae_training_{group_name}_renorm",
        f"{group_name}_holdout_renorm",
        f"{group_name}_holdout_swapped_renorm",
    ]:
        expected.append(f"{base}.h5ad")
        expected.append(f"{base}_v07.h5ad")

missing = [f for f in expected if f not in files_in_dir]
if missing:
    print(f"\nMISSING files: {missing}")
else:
    print(f"\nAll {len(expected)} expected files present.")

print("\n--- Quick sanity check on one AE training file ---")
check_ae = sc.read_h5ad(os.path.join(DATASET_DIR, "ae_training_cd8_renorm_v07.h5ad"))
print(f"  shape: {check_ae.shape}")
print(f"  obs columns: {list(check_ae.obs.columns)}")
print(f"  X dtype: {check_ae.X.dtype}")
ae_vals = check_ae.X if not sp_sparse.issparse(check_ae.X) else check_ae.X.data
print(f"  X min / max: {float(np.min(ae_vals)):.4f} / {float(np.max(ae_vals)):.4f}   (expect [0, <=10])")

print("\n--- Quick sanity check on one holdout file (swapped CellOT data) ---")
check_ho = sc.read_h5ad(os.path.join(DATASET_DIR, "cd8_holdout_renorm_v07.h5ad"))
print(f"  shape: {check_ho.shape}")
print(f"  condition values: {dict(check_ho.obs['condition'].value_counts())}")

print("\n--- Quick sanity check on one swapped holdout file (Normal CellOT data) ---")
check_sw = sc.read_h5ad(os.path.join(DATASET_DIR, "cd8_holdout_swapped_renorm_v07.h5ad"))
print(f"  shape: {check_sw.shape}")
print(f"  condition values: {dict(check_sw.obs['condition'].value_counts())}")
print(f"  species values:   {dict(check_sw.obs['species'].value_counts())}")

print("\n--- Next steps ---")
print("  1. Submit training jobs via: bash sbatch/train/renorm/submit_all_renorm.sh")
print("  2. scGen (stage 1) jobs run first; swapped CellOT + Normal CellOT wait on each scGen.")
print("  3. Logs: cellot/cellot_gpu/results/renorm_<group>/<model>/train_<jobid>.out")

Files in output directory:
  ae_training_cd4_renorm.h5ad                                   407.6 MB
  ae_training_cd4_renorm_v07.h5ad                               407.6 MB
  ae_training_cd8_renorm.h5ad                                   407.6 MB
  ae_training_cd8_renorm_v07.h5ad                               407.6 MB
  ae_training_cd8_thymo_renorm.h5ad                             402.0 MB
  ae_training_cd8_thymo_renorm_v07.h5ad                         402.0 MB
  ae_training_tcell_subtypes_renorm.h5ad                        394.2 MB
  ae_training_tcell_subtypes_renorm_v07.h5ad                    394.2 MB
  cd4_holdout_renorm.h5ad                                        50.6 MB
  cd4_holdout_renorm_v07.h5ad                                    50.6 MB
  cd4_holdout_swapped_renorm.h5ad                                50.6 MB
  cd4_holdout_swapped_renorm_v07.h5ad                            50.6 MB
  cd8_holdout_renorm.h5ad                                        50.6 MB
  cd8_holdout_renorm_v07